In [ ]:
import os
import numpy as np
import sys
sys.path.insert(0, os.path.abspath(".."))
import config

print("Libraries loaded")
print(f"Landmarks path: {config.LANDMARKS_PATH}")

# Check current class sizes
print("\nCurrent class sizes:")
class_sizes = {}
for cls in sorted(os.listdir(config.LANDMARKS_PATH)):
    if cls.endswith('.npy'):
        data = np.load(os.path.join(config.LANDMARKS_PATH, cls))
        class_sizes[cls.replace('.npy','')] = len(data)
        print(f"  {cls.replace('.npy',''):10s}: {len(data):5d}")

print(f"\nMin: {min(class_sizes.values())} — {min(class_sizes, key=class_sizes.get)}")
print(f"Max: {max(class_sizes.values())} — {max(class_sizes, key=class_sizes.get)}")
print(f"Target after augmentation: {max(class_sizes.values())} per class")

Libraries loaded
Landmarks path: C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\data\processed\landmarks

Current class sizes:
  A         :  8458
  B         :  8309
  C         :  8146
  D         :  7629
  E         :  7744
  F         :  8031
  G         :  7844
  H         :  7906
  I         :  7953
  J         :  7503
  K         :  7876
  L         :  7939
  M         :  7900
  N         :  7932
  O         :  8140
  P         :  7601
  Q         :  7954
  R         :  8021
  S         :  8109
  T         :  8054
  U         :  8023
  V         :  7597
  W         :  7787
  X         :  8093
  Y         :  8178
  Z         :  7410
  del       :  6836
  nothing   :  3030
  space     :  7071

Min: 3030 — nothing
Max: 8458 — A
Target after augmentation: 8458 per class


In [ ]:
def augment_flip(landmarks):
    """Mirror hand horizontally — flip x coordinates"""
    augmented = landmarks.copy()
    for i in range(0, len(landmarks), 3):
        augmented[i] = 1.0 - landmarks[i]  # flip x
    return augmented

def augment_scale(landmarks, scale_range=(0.8, 1.2)):
    """Slightly resize the hand"""
    scale = np.random.uniform(*scale_range)
    augmented = landmarks.copy()
    for i in range(0, len(landmarks), 3):
        augmented[i] = landmarks[i] * scale      # scale x
        augmented[i+1] = landmarks[i+1] * scale  # scale y
    return np.clip(augmented, 0.0, 1.0)

def augment_noise(landmarks, noise_level=0.01):
    """Add tiny random variations"""
    noise = np.random.normal(0, noise_level, landmarks.shape)
    return np.clip(landmarks + noise, 0.0, 1.0)

def augment_rotate(landmarks, angle_range=(-15, 15)):
    """Slightly rotate hand landmarks around center"""
    angle = np.random.uniform(*angle_range)
    rad = np.deg2rad(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    augmented = landmarks.copy()
    cx, cy = 0.5, 0.5  # rotate around center
    for i in range(0, len(landmarks), 3):
        x = landmarks[i] - cx
        y = landmarks[i+1] - cy
        augmented[i]   = cos_a * x - sin_a * y + cx
        augmented[i+1] = sin_a * x + cos_a * y + cy
    return np.clip(augmented, 0.0, 1.0)

print("Augmentation functions defined")
print("  - Flip: mirrors hand horizontally")
print("  - Scale: resizes hand by 80-120%")
print("  - Noise: adds tiny random variations")
print("  - Rotate: rotates hand by -15 to +15 degrees")

Augmentation functions defined
  - Flip: mirrors hand horizontally
  - Scale: resizes hand by 80-120%
  - Noise: adds tiny random variations
  - Rotate: rotates hand by -15 to +15 degrees


In [ ]:
from tqdm.notebook import tqdm

# Target — match the largest class
TARGET = max(class_sizes.values())  # 8458
AUGMENTED_PATH = os.path.join(config.BASE_DIR, "data", "processed", "landmarks_augmented")
os.makedirs(AUGMENTED_PATH, exist_ok=True)

augment_funcs = [augment_flip, augment_scale, augment_noise, augment_rotate]

print(f"Target samples per class: {TARGET}")
print(f"Saving to: {AUGMENTED_PATH}")
print("-" * 50)

for cls in tqdm(sorted(class_sizes.keys()), desc="Augmenting classes"):
    # Load original data
    original = np.load(os.path.join(config.LANDMARKS_PATH, f"{cls}.npy"))
    current = list(original)
    needed = TARGET - len(original)

    # Augment until we reach target
    idx = 0
    while len(current) < TARGET:
        sample = original[idx % len(original)]
        func = augment_funcs[idx % len(augment_funcs)]
        current.append(func(sample))
        idx += 1

    # Save augmented class
    result = np.array(current[:TARGET], dtype=np.float32)
    np.save(os.path.join(AUGMENTED_PATH, f"{cls}.npy"), result)

print("-" * 50)
print("Augmentation complete")
print(f"\nVerifying augmented dataset:")
total = 0
for cls in sorted(os.listdir(AUGMENTED_PATH)):
    if cls.endswith('.npy'):
        data = np.load(os.path.join(AUGMENTED_PATH, cls))
        total += len(data)
        print(f"  {cls.replace('.npy',''):10s}: {len(data):5d}")

print(f"\nTotal samples: {total:,}")
print(f"Expected:      {TARGET * 29:,}")

Target samples per class: 8458
Saving to: C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\data\processed\landmarks_augmented
--------------------------------------------------


Augmenting classes:   0%|          | 0/29 [00:00<?, ?it/s]

--------------------------------------------------
Augmentation complete

Verifying augmented dataset:
  A         :  8458
  B         :  8458
  C         :  8458
  D         :  8458
  E         :  8458
  F         :  8458
  G         :  8458
  H         :  8458
  I         :  8458
  J         :  8458
  K         :  8458
  L         :  8458
  M         :  8458
  N         :  8458
  O         :  8458
  P         :  8458
  Q         :  8458
  R         :  8458
  S         :  8458
  T         :  8458
  U         :  8458
  V         :  8458
  W         :  8458
  X         :  8458
  Y         :  8458
  Z         :  8458
  del       :  8458
  nothing   :  8458
  space     :  8458

Total samples: 245,282
Expected:      245,282

Step 07 complete — ready for Step 08 model architecture
